# Lab 15-02 · Transfer Functions and Block Diagrams

Four parts, each built on `blocks.py`. **Predict first**, then run.

In [1]:
import math
import random
import shutil
import subprocess
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from blocks import TF, feedback, tf_from_ss, mason, solve_graph, second_order_specs

## Part 1 · State space to transfer functions

*Predict:* do $\Omega/V$ and $I/V$ of Example 15-02.2's motor share poles? Where is the zero of $I/V$?

In [2]:
R, L, K, J, B = 2.0, 1e-3, 0.05, 2e-5, 1e-5
A = np.array([[-R / L, -K / L], [K / J, -B / J]])
Bv = [1 / L, 0]
G_speed = tf_from_ss(A, Bv, [0, 1])
G_curr = tf_from_ss(A, Bv, [1, 0])
print("Omega/V:", G_speed, " DC gain", round(G_speed.dcgain(), 3), "rad/s per V")
print("I/V:    ", G_curr, " DC gain", round(G_curr.dcgain() * 1e3, 3), "mA/V")
print("poles  ", np.sort(G_speed.poles()), np.sort(G_curr.poles()))
print("zero of I/V", G_curr.zeros(), " eigenvalues of A", np.sort(np.linalg.eigvals(A)))
assert np.allclose(np.sort(G_speed.poles()), np.sort(np.linalg.eigvals(A)))

Omega/V: TF(num=[2500000.0], den=[1.0, 2000.5, 126000.0])  DC gain 19.841 rad/s per V
I/V:     TF(num=[1000.0, 500.0], den=[1.0, 2000.5, 126000.0])  DC gain 3.968 mA/V
poles   [-1935.39708068   -65.10291932] [-1935.39708068   -65.10291932]
zero of I/V [-0.5]  eigenvalues of A [-1935.39708068+0.j   -65.10291932+0.j]


## Part 2 · Two loops, in Python and in ngspice

*Predict:* the overshoot of Example 15-02.3's loop without the inner velocity loop.

In [3]:
G1 = TF([10], [1, 1])
G2 = TF([1], [1, 0])
T = feedback(G1 * feedback(G2, 2), 1)
T_no_inner = feedback(G1 * G2, 1)
print("with inner loop   ", T, " DC", round(T.dcgain(), 4))
print("without inner loop", T_no_inner, " DC", round(T_no_inner.dcgain(), 4))
t = np.linspace(0, 15, 15001)
y, y0 = T.step(t), T_no_inner.step(t)
yf = T.dcgain()
k = np.argmax(y)
out = np.flatnonzero(np.abs(y - yf) > 0.02 * yf)
print(f"peak {y[k]:.4f} at {t[k]:.3f} s, overshoot {100 * (y[k] / yf - 1):.1f}%, 2% settling {t[out[-1] + 1]:.2f} s")
print(f"without inner loop: overshoot {100 * (y0.max() / T_no_inner.dcgain() - 1):.1f}%")
fig, ax = plt.subplots(figsize=(6, 3.2))
ax.plot(t, y, label="with velocity loop")
ax.plot(t, y0, label="without")
ax.set_xlabel("t (s)"); ax.set_ylabel("y"); ax.legend(); ax.grid(True, alpha=0.3)
plt.show()

with inner loop    TF(num=[10.0], den=[1.0, 3.0, 12.0])  DC 0.8333
without inner loop TF(num=[10.0], den=[1.0, 1.0, 10.0])  DC 1.0
peak 1.0176 at 1.006 s, overshoot 22.1%, 2% settling 2.42 s
without inner loop: overshoot 60.5%


In [4]:
if shutil.which("ngspice"):
    res = subprocess.run(["ngspice", "-b", "two_loop.cir"], capture_output=True, text=True, timeout=120)
    print("\n".join(l for l in res.stdout.splitlines() if l.split("=")[0].strip() in ("yfin", "ypk", "tpk", "os")))
else:
    print("ngspice not installed: skip (CI runs two_loop.cir through tools/run_spice.py)")

yfin                =  8.333333e-01
ypk                 =  1.017578e+00 at=  1.006205e+00
tpk                 =  1.006616e+00
yfin = 8.333333e-01
ypk = 1.017578e+00
tpk = 1.006616e+00
os = 2.210936e+01


## Part 3 · Mason's gain formula against a direct solve

*Predict:* how many loops does Example 15-02.5's graph have, and which pairs do not touch?

In [5]:
g = dict(G1=2, G2=5, G3=4, G4=1, G5=3, H1=0.5, H2=0.25, H3=0.1)
E = [("R", "x1", 1), ("x1", "x2", g["G1"]), ("x2", "x1", -g["H1"]), ("x2", "x3", g["G2"]),
     ("x3", "x4", g["G3"]), ("x4", "x3", -g["H2"]), ("x4", "x5", g["G4"]), ("x5", "x2", -g["H3"]),
     ("x2", "x4", g["G5"])]
T5, info = mason(E, "R", "x5", detail=True)
for nodes, gain in info["loops"]:
    print("loop", "->".join(nodes), "gain", round(gain, 3))
pairs = [(a, b) for i, (a, _) in enumerate(info["loops"]) for (b, _) in info["loops"][i + 1:] if set(a).isdisjoint(b)]
print("non-touching pairs:", pairs)
print("Delta =", info["delta"], " Mason T =", round(T5, 4), " direct solve =", round(solve_graph(E, "R", "x5"), 4))

loop x1->x2 gain -1.0
loop x2->x3->x4->x5 gain -2.0
loop x2->x4->x5 gain -0.3
loop x3->x4 gain -1.0
non-touching pairs: [(['x1', 'x2'], ['x3', 'x4'])]
Delta = 6.3  Mason T = 7.3016  direct solve = 7.3016


In [6]:
rng = random.Random(1)
worst = 0.0
for trial in range(200):
    n = rng.randint(4, 7)
    nodes = [f"n{i}" for i in range(n)]
    edges = [(nodes[i], nodes[i + 1], rng.uniform(0.2, 2.0)) for i in range(n - 1)]
    for _ in range(rng.randint(2, 7)):
        a, b = rng.sample(range(1, n), 2)
        edges.append((nodes[a], nodes[b], rng.uniform(-0.6, 0.6)))
    d = solve_graph(edges, nodes[0], nodes[-1])
    worst = max(worst, abs(mason(edges, nodes[0], nodes[-1]) - d) / max(1.0, abs(d)))
print(f"largest relative difference over 200 random graphs: {worst:.1e}")

largest relative difference over 200 random graphs: 9.2e-15


## Part 4 · A PLL's loop-filter resistor

*Predict:* at $R$ = 10 kΩ, where $\zeta > 1$, does the divided phase overshoot after a reference phase step?

In [7]:
Icp, Kv, N, C = 100e-6, 2 * math.pi * 100e6, 100, 1e-9
wn = math.sqrt(Icp * Kv / (2 * math.pi * N * C))
t = np.linspace(0, 6e-5, 60001)
fig, ax = plt.subplots(figsize=(6, 3.2))
for Rf in (2.5e3, 5e3, 10e3):
    zeta = Rf * C * wn / 2
    T = TF([wn**2 * Rf * C, wn**2], [1, wn**2 * Rf * C, wn**2])
    y = T.step(t)
    print(f"R = {Rf / 1e3:4.1f} kOhm: zeta = {zeta:.3f}, overshoot = {100 * (y.max() - 1):.2f}%")
    ax.plot(t * 1e6, y, label=f"R = {Rf / 1e3:g} kΩ")
ax.set_xlabel("t (µs)"); ax.set_ylabel("divided phase / reference step"); ax.legend(); ax.grid(True, alpha=0.3)
plt.show()

R =  2.5 kOhm: zeta = 0.395, overshoot = 36.71%


R =  5.0 kOhm: zeta = 0.791, overshoot = 18.24%


R = 10.0 kOhm: zeta = 1.581, overshoot = 6.97%


**Discussion.** Answer the four questions in the [lab handout](README.md) before reading its answers.